In [16]:
!pip install --upgrade numpy statsmodels arch scikit-learn pandas matplotlib keras-tuner

In [17]:
from data import DataCleaner
import pandas as pd

In [35]:
df = pd.read_csv('aapl_intraday_data_cleaned.csv')["Close"]
    
df.head()

0    238.0000
1    238.0000
2    237.9500
3    237.9106
4    237.9900
Name: Close, dtype: float64

In [36]:
def train_test_split_last_n_days(df: pd.DataFrame, n_days: int = 7):
    """
    Splits a DataFrame into train and test sets,
    where the test set is the last `n_days` calendar days of the index.

    Parameters
    ----------
    df     : pd.DataFrame
        Must have a DateTimeIndex (can be tz-aware or naive).
    n_days : int
        Number of days to reserve for the test set (default=7).

    Returns
    -------
    train_df, test_df : (pd.DataFrame, pd.DataFrame)
    """
    # Ensure sorted by time
    df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True)
    df_sorted = df.set_index("Datetime").sort_index()
    print(df_sorted)
    # Compute cutoff timestamp
    last_ts = df_sorted.index.max()
    print(last_ts)
    
    cutoff  = last_ts - pd.Timedelta(days=n_days)

    # Split
    train_df = df_sorted.loc[df_sorted.index <= cutoff]
    test_df  = df_sorted.loc[df_sorted.index >  cutoff]

    return train_df, test_df
train_rv, test_set = train_test_split_last_n_days(df, n_days=7)
print(train_rv)
print(f"Train from {train_rv.index.min()} to {train_rv.index.max()} ({len(train_rv)} rows)")
print(f"Test  from {test_set.index.min()} to {test_set.index.max()} ({len(test_set)} rows)")
training_set = train_rv.shift(-1).dropna()   # length N−1, target at t is RV at t+1

KeyError: 'Datetime'

In [20]:
# %%
# ── Fix the 2D shape error when scaling a Series ─────────────────────────────
import numpy as np
# assume `training_set` is a pd.Series of shape (n_samples,)
# convert to a 2D array of shape (n_samples, 1)
train_vals = training_set.values.reshape(-1, 1)


# now scale
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
training_scaled = scaler.fit_transform(train_vals)

# ── build LSTM sequences ────────────────────────────────────────────────────
memory = 60  # or whatever lookback window you’re using

X_train, y_train = [], []
for i in range(memory, len(training_scaled)):
    X_train.append(training_scaled[i - memory : i, 0])
    y_train.append(training_scaled[i, 0])

# reshape into [samples, time_steps, features]
X_train = np.array(X_train).reshape(-1, memory, 1)
y_train = np.array(y_train)
 
print(f"Shapes → X_train: {X_train.shape}, y_train: {y_train.shape}")


Shapes → X_train: (4065060, 60, 1), y_train: (4065060,)


In [24]:
# %%
# Cell 7 — Faster Hyperparameter Tuning with Keras Tuner (smaller search space)
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Conv1D, MaxPooling1D
import tensorflow as tf # This code has been tested with TensorFlow 1.6

# 1) Build a much smaller hypermodel factory
def build_lstm_model_fast(hp):
    model = Sequential()
    # only 1–2 LSTM layers
    for i in range(hp.Int('n_layers', 1, 2)):
        units     = hp.Int(f'units_{i}', min_value=16, max_value=64, step=16)
        dropout   = hp.Float(f'dropout_{i}', 0.0, max_value=0.3, step=0.1)
        return_seq = (i < hp.get('n_layers') - 1)
        if i == 0:
            model.add(LSTM(units, return_sequences=return_seq,
                           input_shape=(memory, 1)))
        else:
            model.add(LSTM(units, return_sequences=return_seq))
        model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Float('lr', 1e-3, 1e-2, sampling='log')
        ),
        loss='mean_squared_error'
    )
    return model

# 2) Instantiate tuner with fewer trials
tuner_fast = kt.RandomSearch(
    build_lstm_model_fast,
    objective='val_loss',
    max_trials=5,               # reduce from 20 → 5
    executions_per_trial=1,
    directory='lstm_tuning_fast',
    project_name='stock_pred_fast'
)

# 3) Run the search with fewer epochs for quick feedback
tuner_fast.search(
    X_train, y_train,
    epochs=10,                  # reduce from 50 → 10
    batch_size=4096,              # smaller batch
    validation_split=0.2,        
    callbacks=[EarlyStopping(patience=3)]
)

# 4) Retrieve best model
best_fast = tuner_fast.get_best_models(num_models=1)[0]
best_fast.summary()
model = best_fast

Trial 5 Complete [00h 11m 59s]
val_loss: 6.480437377831549e-07

Best val_loss So Far: 2.6269532327205525e-07
Total elapsed time: 05h 05m 23s


C:\Users\moina\anaconda3\envs\TradingSystem\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 60, 32)              │           4,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 60, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 16)                  │           3,136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,505 (29.32 KB)

 Trainable params: 7,505 (29.32 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
print(df)

                       Datetime      Open      High       Low     Close  \
0     2025-09-03 23:44:00+00:00  238.0000  238.0000  237.9900  238.0000   
1     2025-09-03 23:45:00+00:00  238.0000  238.0000  238.0000  238.0000   
2     2025-09-03 23:46:00+00:00  237.9500  237.9500  237.9500  237.9500   
3     2025-09-03 23:47:00+00:00  237.9997  237.9997  237.9106  237.9106   
4     2025-09-03 23:48:00+00:00  237.9200  238.0000  237.9200  237.9900   
...                         ...       ...       ...       ...       ...   
88095 2026-03-02 19:30:00+00:00  266.0450  266.1000  265.9300  265.9900   
88096 2026-03-02 19:31:00+00:00  265.9900  266.0000  265.8300  265.8400   
88097 2026-03-02 19:32:00+00:00  265.8400  266.1000  265.8300  266.0000   
88098 2026-03-02 19:33:00+00:00  266.0100  266.3500  265.9700  266.3500   
88099 2026-03-02 19:34:00+00:00  266.3600  266.4900  266.3200  266.4600   

        Volume  Return_1  LogReturn_1  Return_5  LogReturn_5  ...  \
0        568.0 -0.000336    -0

In [33]:
# 2️⃣ Build the test‐input windows
full_vals = df[].values.reshape(-1, 1)
inputs   = full_vals[-len(test_set) - memory :]
print(inputs)

inputs_scaled = scaler.transform(inputs)

X_test = []
for i in range(memory, len(inputs_scaled)):
    X_test.append(inputs_scaled[i - memory : i, 0])
X_test = np.array(X_test).reshape(-1, memory, 1)

# 3️⃣ Predict & invert scaling
pred_scaled = model.predict(X_test)
pred_rv = scaler.inverse_transform(pred_scaled).flatten()

# 4️⃣ Plot Actual vs. Predicted
plt.figure(figsize=(12, 5))
plt.plot(test_set.index, test_set.values, 'o-', label='Actual RV', linewidth=2)
plt.plot(test_set.index.shift(-1), pred_rv,       's--',label='Predicted RV', linewidth=2) # account for lag
plt.title('Hourly Realized Volatility: Actual vs. Predicted (Last 7 Days)')
plt.xlabel('Timestamp')
plt.ylabel('Realized Volatility')

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



[[-0.0020315262781687]
 [-0.0020335926267133]
 [-0.0021066094872662]
 ...
 [55066.9]
 [1.408395969266474]
 [20665571.76]]


TypeError: float() argument must be a string or a real number, not 'Timestamp'